# Non-Contractual Player Churn Prediction: Multi-Product iGaming Pipeline

## 1. Executive Summary & Problem Formulation
In continuous-play digital gaming platforms, churn is **non-contractual**: players never explicitly cancel an account; they simply disengage, drain balances, or migrate to competing operators.

### 1.1 Objective
Build an end-to-end churn classification pipeline on longitudinal poker and cashier records to identify players at high risk of 45-day activity cessation.

### 1.2 Data Topology
The analysis integrates five relational event streams keyed on `UserID`:
1. **Demographics**: Static user-level attributes (`SystemAgeAsOfReg`, `Gender`, `CountryID`).
2. **Cash Games**: Continuous stake and rake volume (`Windows`, `StakesC`, `WinningsC`).
3. **Tournaments**: Discrete fixed-fee game volume (`Trnmnts`, `StakesT`, `WinningsT`).
4. **Cashier Deposits**: Liquidity inflows and payment gateway health (`Amount`, `Status`, `PayMethCat`).
5. **Cashier Withdrawals**: Capital extraction and cashing-out events (`Amount`, `Status`).

In [23]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
# Loading datasets
Demographics =pd.read_csv('datasets/SecondSeshData1Demographics.csv') 
CashGames = pd.read_csv('datasets/SecondSeshData2CashGames.csv')
Tournaments = pd.read_csv('datasets/SecondSeshData3Tournaments.csv')
Deposits = pd.read_csv('datasets/SecondSeshData4Deposits.csv')
Withdrawals = pd.read_csv('datasets/SecondSeshData5Withdrawals.csv')

In [25]:
# confirm the data types of each date column
print(CashGames.dtypes)

UserID         int64
Date             str
Windows        int64
StakesC      float64
WinningsC    float64
dtype: object


In [26]:
# change date columns to datetime format
for df, columns in (
    (CashGames, ["Date"]),
    (Tournaments, ["Date"]),
    (Deposits, ["SummaryDate", "ProcessDate"]),
    (Withdrawals, ["SummaryDate", "ProcessDate"]),
):
    df[columns] = df[columns].apply(pd.to_datetime)

In [27]:
tables = [
    ("CashGames", CashGames),
    ("Tournaments", Tournaments),
    ("Deposits", Deposits),
    ("Withdrawals", Withdrawals),
]

for name, table in tables:
    print(f"Size of the {name} table is: {table.shape}")

Size of the CashGames table is: (51763, 5)
Size of the Tournaments table is: (82831, 5)
Size of the Deposits table is: (295088, 10)
Size of the Withdrawals table is: (32307, 10)


In [29]:
for name, table in tables:
    print(f'The number of unique players in the {name} table is: {table["UserID"].nunique()}')

The number of unique players in the CashGames table is: 4232
The number of unique players in the Tournaments table is: 3448
The number of unique players in the Deposits table is: 5008
The number of unique players in the Withdrawals table is: 1827


In [30]:
activity_user_ids = pd.concat(
    [
        CashGames["UserID"],
        Tournaments["UserID"],
        Deposits["UserID"],
        Withdrawals["UserID"],
    ],
    ignore_index=True,
).dropna().drop_duplicates()

demographic_user_ids = Demographics["UserID"].dropna().drop_duplicates()

matches = activity_user_ids.isin(demographic_user_ids)
matched_count = matches.sum()
total_count = len(activity_user_ids)

percentage = matched_count / total_count * 100 if total_count else 0

print(f"{matched_count} of {total_count} activity users are in Demographics ({percentage:.2f}%).")

5028 of 5028 activity users are in Demographics (100.00%).
